## Projet ML P1 IA G5 : Segmentation Client

Groupe 5 :

* __Rokhaya Coumba Diouf__
* __Yohan Fredy Hounyouvi__
* __Khadidiatou Gueye__
* __Cheikh Ahmed Tidiane Ciss__

### Segmentation de la clientèle d'un grossiste

#### 0. Introduction

##### 0.1 Contexte métier
Un grossiste portugais en produits alimentaires approvisionne différents types de clients professionnels (B2B) : hôtels, restaurants, cafés, épiceries, supérettes, etc.

Aujourd'hui, il applique la **même stratégie** (logistique, offre commerciale, marketing) à tous ses clients, alors que leurs besoins et leurs volumes d'achat sont très différents.

Toute décision globale, par exemple réduire la fréquence des livraisons pour baisser les coûts, risque donc de **satisfaire certains clients et d'en mécontenter d'autres**. Le grossiste a besoin de mieux connaître sa clientèle pour adapter ses décisions à chaque type de client.

### 0.2 Problématique
> Peut-on identifier des groupes de clients homogènes à partir de leurs habitudes de dépenses, et en tirer des recommandations concrètes pour le grossiste ?

### 0.3 Objectifs
1. Explorer et comprendre le comportement d'achat des clients.
2. Construire une segmentation par **apprentissage non supervisé** (clustering).
3. Déterminer un nombre de segments pertinent et justifié.
4. Interpréter chaque segment (persona) et valider la segmentation.
5. Proposer des recommandations métier adaptées à chaque segment.

### 0.4 Description du jeu de données
**Source :** *Wholesale Customers Data Set*, UCI Machine Learning Repository.
Chaque ligne correspond à un client. Les variables de dépenses sont des **montants annuels** (en unités monétaires).

| Variable | Type | Description |
|---|---|---|
| `Channel` | Catégorielle | Canal de vente : 1 = Horeca (Hôtels/Restaurants/Cafés), 2 = Retail (commerce de détail) |
| `Region` | Catégorielle | Région : 1 = Lisbonne, 2 = Porto, 3 = Autre |
| `Fresh` | Numérique | Dépenses annuelles en produits frais |
| `Milk` | Numérique | Dépenses annuelles en produits laitiers |
| `Grocery` | Numérique | Dépenses annuelles en épicerie |
| `Frozen` | Numérique | Dépenses annuelles en surgelés |
| `Detergents_Paper` | Numérique | Dépenses annuelles en détergents et produits papier |
| `Delicassen` | Numérique | Dépenses annuelles en produits traiteur / épicerie fine |

### 0.5 Démarche générale
Le dataset ne contient **aucune variable cible** : il s'agit donc d'un problème d'**apprentissage non supervisé**.

**Choix méthodologique :** la segmentation sera construite uniquement à partir des **6 variables de dépenses**. Les variables `Channel` et `Region` seront mises de côté pour servir à **l'interprétation et à la validation** des segments. Si nos clusters retrouvent la distinction Horeca/Retail sans l'avoir vue pendant l'apprentissage, cela confirmera que la segmentation capte une structure réelle des données.

**Étapes du projet :**
1. Imports et chargement des données
2. Analyse exploratoire (EDA)
3. Prétraitement
4. Réduction de dimension (ACP)
5. Modélisation (K-Means, CAH, GMM)
6. Interprétation des segments
7. Evaluation et Validation
8. Affectation d'un nouveau client
9. Recommandations métier et conclusion

### 1. Le Machine Learning non supervisé

#### 1.1 Supervisé vs non supervisé
En Machine Learning, on distingue deux grandes familles d'apprentissage :

| | Apprentissage supervisé | Apprentissage non supervisé |
|---|---|---|
| **Données** | Variables explicatives **+ une variable cible** (étiquette) | Variables explicatives **uniquement**, sans étiquette |
| **Objectif** | Apprendre à **prédire** la cible | **Découvrir des structures cachées** dans les données |
| **Exemples de tâches** | Classification, régression | Clustering, réduction de dimension, détection d'anomalies |
| **Évaluation** | Comparaison prédiction / vraie valeur (accuracy, RMSE…) | Pas de « bonne réponse » : métriques internes (silhouette…) et interprétation métier |

#### 1.2 Pourquoi le non supervisé dans notre cas ?
Dans notre dataset, **aucune colonne n'indique à quel segment appartient chaque client** : les segments n'existent pas encore, c'est justement ce qu'on cherche à découvrir.
On ne peut donc pas entraîner un modèle supervisé. On utilise le **clustering**, qui regroupe automatiquement les clients selon la similarité de leurs dépenses.

#### 1.3 Le clustering
Le clustering consiste à partitionner les observations en groupes (**clusters**) tels que :
- les clients d'un **même cluster** soient **le plus similaires possible** (forte homogénéité intra-cluster) ;
- les clients de **clusters différents** soient **le plus différents possible** (forte séparation inter-clusters).

La similarité est généralement mesurée par une **distance** (souvent la distance euclidienne) entre les profils de dépenses. Cela implique que les variables soient **à la même échelle**, d'où l'importance du prétraitement (standardisation).

Principaux algorithmes envisagés dans ce projet :
- **K-Means** : affecte chaque client au centre de cluster le plus proche, en minimisant la variance intra-cluster. Le nombre de clusters *k* doit être choisi à l'avance.
- **Clustering hiérarchique (CAH)** : construit un arbre de regroupements successifs (dendrogramme), utile pour visualiser et choisir le nombre de groupes.
- **Modèles de mélange gaussien (GMM)** : approche probabiliste où chaque client a une probabilité d'appartenance à chaque cluster.

#### 1.4 Application à la segmentation client
Appliqué à notre problème, le clustering permettra de passer de **440 clients traités de la même façon** à **quelques segments** au comportement d'achat homogène, chacun pouvant faire l'objet d'une stratégie commerciale et logistique adaptée.

### Étape 3 : Chargement du dataset avec Pandas

#### 2. Chargement des données

##### 2.1 Import des bibliothèques
**Pourquoi :** on importe les bibliothèques utilisées dans cette partie :
- **Pandas** : manipulation de données tabulaires (DataFrame) ;
- **NumPy** : calcul numérique ;
- **Matplotlib** et **Seaborn** : visualisation.

On fixe aussi une **graine aléatoire** (`RANDOM_STATE`) pour garantir la **reproductibilité** des résultats, car certaines étapes du projet (K-Means, GMM) dépendent d'une initialisation aléatoire.

In [1]:
# --- Manipulation des données ---
import numpy as np
import pandas as pd

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Divers ---
import warnings
warnings.filterwarnings("ignore")

# --- Paramètres globaux ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Style des graphiques
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)

# Affichage pandas : toutes les colonnes, 2 décimales
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


### 2.2 Lecture du fichier CSV
**Pourquoi :** on charge le fichier dans un DataFrame Pandas, puis on affiche les premières lignes pour vérifier que la lecture s'est bien passée (bon séparateur, bons noms de colonnes, valeurs cohérentes).

In [2]:
# Chargement du dataset
df = pd.read_csv("data/Wholesale customers data.csv")

print("Dataset chargé avec succès.")
df.head()

Dataset chargé avec succès.


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185


In [3]:
# Aperçu des dernières lignes et d'un échantillon aléatoire
display(df.tail())
display(df.sample(5, random_state=RANDOM_STATE))

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
435,1,3,29703,12051,16027,13135,182,2204
436,1,3,39228,1431,764,4510,93,2346
437,2,3,14531,15488,30243,437,14841,1867
438,1,3,10290,1981,2232,1038,168,2125
439,1,3,2787,1698,2510,65,477,52


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
265,1,1,5909,23527,13699,10155,830,3636
78,1,3,10766,1175,2067,2096,301,167
347,2,3,27380,7184,12311,2809,4621,1022
255,1,1,31614,489,1495,3242,111,615
327,1,2,542,899,1664,414,88,522


### 2.3 Définition des groupes de variables
**Pourquoi :** on sépare dès maintenant les variables en deux groupes, qu'on réutilisera dans tout le notebook :
- les **variables catégorielles** (`Channel`, `Region`), qui serviront à l'interprétation et à la validation ;
- les **variables de dépenses**, qui serviront à construire la segmentation.

On crée aussi des dictionnaires de correspondance pour remplacer les codes numériques par des libellés lisibles dans les graphiques.

In [4]:
# Groupes de variables
CAT_COLS = ["Channel", "Region"]
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

# Correspondance codes -> libellés
channel_map = {1: "Horeca", 2: "Retail"}
region_map = {1: "Lisbonne", 2: "Porto", 3: "Autre"}

print("Variables catégorielles :", CAT_COLS)
print("Variables de dépenses   :", SPEND_COLS)

Variables catégorielles : ['Channel', 'Region']
Variables de dépenses   : ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']


**Observation :** le fichier est correctement chargé. Les 8 colonnes attendues sont présentes, avec des noms cohérents avec la description du dataset. Les variables de dépenses prennent des valeurs très variables d'un client à l'autre (de quelques unités à plusieurs dizaines de milliers), ce que nous analyserons en détail dans l'exploration.

#### Étape 4 : Exploration de la structure des données

##### 3.1 Dimensions du dataset
**Pourquoi :** connaître le nombre d'observations et de variables permet de juger de la taille du problème et du choix des méthodes. Avec peu de données, par exemple, il faut être prudent sur le nombre de clusters.

In [5]:
print(f"Nombre de lignes (clients)   : {df.shape[0]}")
print(f"Nombre de colonnes (variables) : {df.shape[1]}")

Nombre de lignes (clients)   : 440
Nombre de colonnes (variables) : 8


**Observation :** le dataset contient **440 clients** décrits par **8 variables**. C'est un jeu de données de petite taille : les algorithmes s'exécuteront rapidement, mais chaque valeur extrême aura un poids important sur les résultats.

### 3.2 Types de variables
**Pourquoi :** vérifier que chaque colonne a le type attendu (numérique ou catégoriel). Une variable mal typée peut fausser les calculs et les graphiques.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    int64
 7   Delicassen        440 non-null    int64
dtypes: int64(8)
memory usage: 27.6 KB


In [7]:
# Tableau récapitulatif : type pandas et nature réelle de chaque variable
types_df = pd.DataFrame({
    "Type pandas": df.dtypes,
    "Nature": ["Catégorielle" if col in CAT_COLS else "Numérique continue" for col in df.columns]
})
types_df

,Type pandas,Nature
Channel,int64,Catégorielle
Region,int64,Catégorielle
Fresh,int64,Numérique continue
Milk,int64,Numérique continue
Grocery,int64,Numérique continue
Frozen,int64,Numérique continue
Detergents_Paper,int64,Numérique continue
Delicassen,int64,Numérique continue


**Observation :**
- Les 8 variables sont stockées en **entiers (`int64`)**.
- Attention : `Channel` et `Region` sont codées en chiffres, mais ce sont des **variables catégorielles** (nominales). Leurs valeurs n'ont pas de sens numérique : la région 3 n'est pas « plus grande » que la région 1. Elles ne doivent donc **pas être traitées comme des nombres** dans le clustering.
- Les 6 variables de dépenses sont des **variables quantitatives continues** (montants annuels).

#### 3.3 Description statistique
**Pourquoi :** les statistiques descriptives résument l'ordre de grandeur, la dispersion et l'étendue de chaque variable de dépenses. Comparer la **moyenne** et la **médiane** donne un premier indice d'asymétrie : si la moyenne est nettement supérieure à la médiane, quelques valeurs très élevées tirent la moyenne vers le haut.

In [8]:
# Statistiques descriptives des variables de dépenses
stats = df[SPEND_COLS].describe().T
stats["median"] = df[SPEND_COLS].median()
stats["mean/median"] = stats["mean"] / stats["median"]
stats["cv (std/mean)"] = stats["std"] / stats["mean"]   # coefficient de variation
stats

,count,mean,std,min,25%,50%,75%,max,median,mean/median,cv (std/mean)
Fresh,440.00,12000.30,12647.33,3.00,3127.75,8504.00,16933.75,112151.00,8504.00,1.41,1.05
Milk,440.00,5796.27,7380.38,55.00,1533.00,3627.00,7190.25,73498.00,3627.00,1.60,1.27
Grocery,440.00,7951.28,9503.16,3.00,2153.00,4755.50,10655.75,92780.00,4755.50,1.67,1.20
Frozen,440.00,3071.93,4854.67,25.00,742.25,1526.00,3554.25,60869.00,1526.00,2.01,1.58
Detergents_Paper,440.00,2881.49,4767.85,3.00,256.75,816.50,3922.00,40827.00,816.50,3.53,1.65
Delicassen,440.00,1524.87,2820.11,3.00,408.25,965.50,1820.25,47943.00,965.50,1.58,1.85


In [9]:
# Dépense totale par client et part moyenne de chaque catégorie
total = df[SPEND_COLS].sum(axis=1)
print("Dépense annuelle totale par client :")
print(total.describe().round(2), "\n")

part_moyenne = (df[SPEND_COLS].sum() / df[SPEND_COLS].sum().sum() * 100).sort_values(ascending=False)
print("Part de chaque catégorie dans les dépenses totales (%) :")
print(part_moyenne.round(1))

Dépense annuelle totale par client :
count      440.00
mean     33226.14
std      26356.30
min        904.00
25%      17448.75
50%      27492.00
75%      41307.50
max     199891.00
dtype: float64 

Part de chaque catégorie dans les dépenses totales (%) :
Fresh              36.10
Grocery            23.90
Milk               17.40
Frozen              9.20
Detergents_Paper    8.70
Delicassen          4.60
dtype: float64


**Observations :**
- **`Fresh`** est la catégorie qui pèse le plus en moyenne (≈ 12 000), suivie de `Grocery` (≈ 7 950) et `Milk` (≈ 5 800). `Delicassen` est la plus faible (≈ 1 525).
- Pour **toutes** les variables, la **moyenne est nettement supérieure à la médiane** (par exemple `Detergents_Paper` : moyenne ≈ 2 880 contre médiane ≈ 816). Les distributions sont donc **asymétriques à droite** : la majorité des clients dépense peu, et quelques clients dépensent énormément.
- L'**écart-type est très élevé**, souvent du même ordre que la moyenne ou plus (coefficient de variation proche de 1 ou supérieur) : les clients sont **très hétérogènes**, ce qui est plutôt encourageant pour une segmentation.
- L'**étendue est considérable** : des minimums à quelques unités (3 en `Fresh`, `Grocery`, `Detergents_Paper`) et des maximums très élevés (112 151 en `Fresh`, 92 780 en `Grocery`, 60 869 en `Frozen`). Cela signale la présence probable de **valeurs extrêmes**, à étudier.

Ces constats suggèrent qu'une **transformation des données** (par exemple logarithmique) sera nécessaire avant le clustering.

### 3.4 Valeurs manquantes et doublons
**Pourquoi :** les algorithmes de clustering ne gèrent pas les valeurs manquantes, et des doublons donneraient un poids artificiel à certains clients.

In [10]:
missing = pd.DataFrame({
    "Nb manquants": df.isnull().sum(),
    "% manquants": (df.isnull().mean() * 100).round(2)
})
display(missing)

print(f"Nombre de lignes dupliquées : {df.duplicated().sum()}")
print(f"Nombre de dépenses négatives : {(df[SPEND_COLS] < 0).sum().sum()}")

,Nb manquants,% manquants
Channel,0,0.00
Region,0,0.00
Fresh,0,0.00
Milk,0,0.00
Grocery,0,0.00
Frozen,0,0.00
Detergents_Paper,0,0.00
Delicassen,0,0.00


Nombre de lignes dupliquées : 0
Nombre de dépenses négatives : 0


**Observation :** le dataset ne contient **aucune valeur manquante**, **aucun doublon** et **aucune dépense négative**. Aucun traitement d'imputation ou de nettoyage n'est nécessaire à ce stade.

#### 3.5 Valeurs uniques
**Pourquoi :** le nombre de valeurs uniques confirme la nature des variables (peu de modalités = catégorielle, beaucoup = continue) et permet d'étudier la répartition des clients par canal et par région.

In [11]:
# Nombre de valeurs uniques par variable
df.nunique().to_frame("Nb valeurs uniques")

,Nb valeurs uniques
Channel,2
Region,3
Fresh,433
Milk,421
Grocery,430
Frozen,426
Detergents_Paper,417
Delicassen,403


In [12]:
# Répartition des variables catégorielles (effectifs et pourcentages)
for col, mapping in [("Channel", channel_map), ("Region", region_map)]:
    counts = df[col].map(mapping).value_counts()
    pct = (df[col].map(mapping).value_counts(normalize=True) * 100).round(1)
    print(f"--- {col} ---")
    print(pd.DataFrame({"Effectif": counts, "%": pct}), "\n")

# Croisement Channel x Region
pd.crosstab(df["Region"].map(region_map), df["Channel"].map(channel_map), margins=True, margins_name="Total")

--- Channel ---
         Effectif     %
Channel                
Horeca        298 67.70
Retail        142 32.30 

--- Region ---
          Effectif     %
Region                  
Autre          316 71.80
Lisbonne        77 17.50
Porto           47 10.70 



Channel,Horeca,Retail,Total
Region,,,
Autre,211,105,316
Lisbonne,59,18,77
Porto,28,19,47
Total,298,142,440


**Observations :**
- `Channel` a **2 modalités** et `Region` **3 modalités**, ce qui confirme leur nature catégorielle. Les variables de dépenses ont des centaines de valeurs uniques : elles sont bien continues.
- **Canal :** environ **2/3 des clients sont Horeca** (298, soit ≈ 68 %) contre **1/3 Retail** (142, soit ≈ 32 %). Le dataset est déséquilibré en faveur de la restauration et de l'hôtellerie.
- **Région :** la grande majorité des clients se trouve dans la région **« Autre »** (316, soit ≈ 72 %), contre 77 à Lisbonne et 47 à Porto.
- Le croisement montre que la proportion Horeca/Retail est **globalement similaire d'une région à l'autre**.

La région semble peu discriminante. Le canal, en revanche, sera une référence utile pour **valider** nos futurs clusters.